# Ускорение модели с помощью 4-битного квантования

Будем использовать библиотеку `bitsandbytes` (BNB) с параметром `load_in_4bit=True`.
- **Механизм:** уменьшение точности весов с 16 бит (2 байта) до 4 бит (0.5 байта).
- **Ожидаемый результат:** $\approx$ 4-кратное уменьшение размера модели (с ~26 ГБ до ~7-8 ГБ).
- **Компромисс:** возможное незначительное ухудшение перплексии/точности, которое обычно пренебрежимо мало для больших моделей.

In [1]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import time
import json
import psutil
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# конфигурация модели и устройства
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def get_memory_usage():
    """
    Возвращает текущее потребление видеопамяти в ГБ.
    
    Returns:
        float: Объем занятой памяти в ГБ. Если CUDA недоступна (CPU/MPS), 
               возвращает потребление RAM процессом.
    """
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    else:
        # для CPU/MPS используем RSS процесса как приближение
        return psutil.Process().memory_info().rss / 1024**3

print(f"Device: {DEVICE}")

Device: cuda


## Реализация эксперимента
Мы определим общую функцию оценки, которая будет измерять:
1. **Время загрузки (Load Time)**
2. **Пиковое потребление памяти (Peak Memory Usage)**
3. **Задержку инференса (Inference Latency - Tokens per second)**
4. **Качество вывода (Output Quality)** (Качественная проверка задачи на логическое рассуждение)

In [ ]:
def evaluate_model(model, tokenizer, prompt, run_name="Эксперимент"):
    """
    Выполняет инференс модели и собирает ключевые метрики производительности.
    
    Алгоритм:
    1. Замер памяти до генерации.
    2. Прогрев модели - генерация нескольких токенов для исключения эффекта cold start.
    3. Замер пикового потребления памяти и времени генерации.
    4. Декодирование ответа и расчет скорости (токенов в секунду).
    
    Args:
        model: загруженная модель (FP16 или Quantized).
        tokenizer: токенизатор для модели.
        prompt (str): входной запрос для генерации.
        run_name (str): название эксперимента для логирования.

    Returns:
        dict: Словарь с результатами {latency, throughput, memory, output}.
    """
    print(f"\n--- Запуск: {run_name} ---")
    
    # замер памяти перед инференсом
    mem_before = get_memory_usage()
    print(f"Память до инференса: {mem_before:.2f} GB")
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # warmup
    print("Выполняется warm-up...")
    _ = model.generate(**inputs, max_new_tokens=5)
    
    # тест скорости
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
             torch.cuda.reset_peak_memory_stats(i)
        
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=450,
            do_sample=True,
        )
    
    end_time = time.time()
    
    # расчет метрик
    latency = end_time - start_time
    
    # считаем только сгенерированные токены (без входных)
    new_tokens = len(outputs[0]) - len(inputs.input_ids[0])
    throughput = new_tokens / latency
    
    # пиковое потребление памяти во время генерации
    if torch.cuda.is_available():
        peak_mem = 0
        for i in range(torch.cuda.device_count()):
            peak_mem += torch.cuda.max_memory_allocated(i)
        peak_mem = peak_mem / 1024**3
    else:
        peak_mem = get_memory_usage()
    
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Время выполнения (Latency): {latency:.2f} сек")
    print(f"Скорость (Throughput): {throughput:.2f} токенов/сек")
    print(f"Пиковое потребление памяти (VRAM): {peak_mem:.2f} GB")
    print(f"Пример ответа: {output_text}...")
    
    return {
        "run_name": run_name,
        "latency_sec": latency,
        "throughput_tps": throughput,
        "peak_memory_gb": peak_mem,
        "output": output_text
    }

## Загрузка базовой модели (FP16)
Поскольку запуск базовой модели на одной карте невозможен (OOM), мы загрузим результаты, полученные в ходе профилирования на двух картах T4.
Результаты хранятся в файле `src/profiling_results.json`.

In [4]:
import json
import os

baseline_path = "/kaggle/input/datasets/kianayose/profiling-results/profiling_results.json"
baseline_data = {}

try:
    with open(baseline_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        baseline_data = {
            "vram_gb": sum([g["used_gb"] for g in data["memory_static"]]), # Суммируем по двум GPU
            "throughput": data["performance"]["throughput_tps"],
            "latency": data["performance"]["latency_total_sec"]
        }
        print(f"Загружены данные базовой модели из {baseline_path}")
except FileNotFoundError:
    print("Файл результатов профилирования не найден. Используем значения из отчета ДЗ №1")
    baseline_data = {
        "vram_gb": 27.5,
        "throughput": 2.18,
        "latency": 206.0
    }

Загружены данные базовой модели из /kaggle/input/datasets/kianayose/profiling-results/profiling_results.json


In [5]:
print("\n" + "="*50)
print(f"Запуск эксперимента: DeepSeek-R1-Distill-Qwen-14B (4-bit)")
print("="*50)

res_4bit = None
# запуск инференса
try:
    # очистка памяти для чистоты эксперимента
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print("Загрузка 4-битной модели с NF4...")
    start_load = time.time()
    
    # конфигурация квантования
    # load_in_4bit=True: загрузка весов в 4 бита
    # bnb_4bit_compute_dtype=torch.float16: деквантование происходит в FP16 для вычислений
    # bnb_4bit_quant_type="nf4": формат NormalFloat4
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", 
        bnb_4bit_use_double_quant=True
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    max_memory_mapping = {0: "10GB", 1: "10GB"}
    
    model_4bit = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory=max_memory_mapping,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        offload_folder="offload",
        trust_remote_code=True
    )
    
    load_time = time.time() - start_load
    print(f"Квантованная модель загружена за: {load_time:.2f} сек")
    
    # тестовый промпт
    prompt = "Реши уравнение: x^2 - 5x + 6 = 0 по шагам"
    # запуск эксперимента
    res_4bit = evaluate_model(model_4bit, tokenizer, prompt, run_name="NF4")
    
    with open("quantization_results.json", "w") as f:
        json.dump(res_4bit, f, indent=4)
    print("Результаты сохранены в quantization_results.json")

except Exception as e:
    print(f"\nОшибка при квантизации модели: {e}")
    if not torch.cuda.is_available():
        print("CUDA недоступна. Используем симуляцию результатов для демонстрации кода сравнения.")
        res_4bit = {
            "peak_memory_gb": 9.2,
            "throughput_tps": 10.5,
            "latency_sec": 42.0
        }


Запуск эксперимента: DeepSeek-R1-Distill-Qwen-14B (4-bit)
Загрузка 4-битной модели с NF4...


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Квантованная модель загружена за: 216.41 сек

--- Запуск: NF4 ---
Память до инференса: 2.78 GB
Выполняется warm-up...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Время выполнения (Latency): 41.07 сек
Скорость (Throughput): 6.77 токенов/сек
Пиковое потребление памяти (VRAM): 9.59 GB
Пример ответа: Реши уравнение: x^2 - 5x + 6 = 0 по шагам.

</think>

Решение уравнения \( x^2 - 5x + 6 = 0 \) по шагам:

**1. Написать уравнение:**
\[ x^2 - 5x + 6 = 0 \]

**2. Использовать метод разложения на множители:**
Мы ищем два числа, которые перемножаются на 6 и прибавляются до 5.

**3. Найти множители:**
Два числа 2 и 3:
\[ 2 \times 3 = 6 \]
\[ 2 + 3 = 5 \]

**4. Разложить уравнение:**
\[ (x - 2)(x - 3) = 0 \]

**5. Решить уравнение:**
Для этого приравниваем каждый множитель к нулю:
\[ x - 2 = 0 \quad \Rightarrow \quad x = 2 \]
\[ x - 3 = 0 \quad \Rightarrow \quad x = 3 \]

**6. Записать ответ:**
Решения уравнения:
\[ x = 2 \quad \text{и} \quad x = 3 \]...
Результаты сохранены в quantization_results.json


## Сравнение результатов

In [8]:
if res_4bit and baseline_data:
    w1, w2, w3, w4 = 20, 25, 30, 15

    print("\n" + "=" * (w1 + w2 + w3 + w4 + 10))
    print(f"{'Метрика':<{w1}} | {'Базовая модель (FP16)':<{w2}} | {'Квантованная модель (4-bit)':<{w3}} | {'Изменение'}")
    print("-" * (w1 + w2 + w3 + w4 + 10))

    # память
    b_mem = baseline_data["vram_gb"]
    q_mem = res_4bit["peak_memory_gb"]
    mem_delta = ((q_mem - b_mem) / b_mem) * 100
    print(f"{'VRAM (ГБ)':<{w1}} | {b_mem:<{w2}.2f} | {q_mem:<{w3}.2f} | {mem_delta:.1f}%")

    # скорость
    b_speed = baseline_data["throughput"]
    q_speed = res_4bit["throughput_tps"]
    speed_factor = q_speed / b_speed if b_speed > 0 else 0
    print(f"{'Скорость (ток/с)':<{w1}} | {b_speed:<{w2}.2f} | {q_speed:<{w3}.2f} | x{speed_factor:.1f}")
    
    # Latency
    b_lat = baseline_data["latency"]
    q_lat = res_4bit["latency_sec"]
    lat_delta = ((q_lat - b_lat) / b_lat) * 100
    print(f"{'Задержка (с)':<{w1}} | {b_lat:<{w2}.2f} | {q_lat:<{w3}.2f} | {lat_delta:.1f}%")


Метрика              | Базовая модель (FP16)     | Квантованная модель (4-bit)    | Изменение
----------------------------------------------------------------------------------------------------
VRAM (ГБ)            | 26.06                     | 9.59                           | -63.2%
Скорость (ток/с)     | 2.18                      | 6.77                           | x3.1
Задержка (с)         | 206.26                    | 41.07                          | -80.1%


## Результаты эксперимента

**Ключевые наблюдения:**
1.  **VRAM:** объем используемой видеопамяти сократился ~ в 2.5 раза. Теперь модель может быть развернута на доступных GPU (например, RTX 3080/4070 или одной T4), в то время как базовая версия требовала двух карт T4 (или одной A100/V100 32GB).
2.  **Скорость:** кратный прирост скорости (с ~2 до ~7 токенов/сек) – комфортное UI для пользователя в режиме реального времени.
3.  **Качество:** визуальная оценка сгенерированных ответов показала, что логика рассуждений и точность решения сохранились.